In [ ]:
import joblib
import pandas as pd
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier # Crucial for multi-label
from sklearn.metrics import classification_report, f1_score, make_scorer
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Initialize tools
nltk.download('stopwords')
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Some constants
prefix = ""
model_path = "../resources/models"
logistic_regression_pkl = f"{prefix}logistic_regression.pkl"
naive_bayes_pkl = f"{prefix}naive_bayes.pkl"
vectorizer_pkl = f"{prefix}text_vectorizer.pkl"
dataset = 'synonym_youtoxic_english_1000.csv'
file_path = f"../resources/dataset/{dataset}"

df = pd.read_csv(file_path)

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize and remove stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization (reducing words to their base or root form)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens back into a single string
    return ' '.join(tokens)


###########################################################################################

# 1. Clean Text data
print("Cleaning text...")
df['Cleaned_Text'] = df['Text'].apply(preprocess_text)



# 2. Prepare X and y
df_work = df.drop_duplicates(subset=['Text'])
X = df_work['Cleaned_Text']

target_cols = [
    'IsToxic', 
    'IsAbusive', 
    'IsProvocative', 
    'IsObscene', 
    'IsHatespeech', 
    'IsRacist',
    # 'IsThreat',
    # 'IsReligiousHate',
    # 'IsNationalist'
]
y = df_work[target_cols]

# Split the data into development data and test data. Test data will be used at the very end
# to prove the efficiency of the model
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Split the development data into train data and validation data. This is the data that will be
# used to train and validate the moded
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, random_state=42
)

# tfidf_vectorizer = TfidfVectorizer(max_features=10000, analyzer='char', ngram_range=(1, 2))
tfidf_vectorizer = TfidfVectorizer(max_features=10000, min_df=10, stop_words='english', ngram_range=(1, 2))

# Create vectorized versions of the train, validation and test data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf = tfidf_vectorizer.transform(X_val)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Train logistic regression base model
print("\n*** Train base Logistic Regression ***")
micro_f1_scorer = make_scorer(f1_score, average='micro')

base_lr = LogisticRegression(
    solver='liblinear',
    random_state=42,
    # Crucial for stability in imbalanced data:
    class_weight='balanced', 
    max_iter=50000 
)

moc = MultiOutputClassifier(base_lr, n_jobs=-1)
moc.fit(X_train_tfidf, y_train)

# Predict using the train data
y_train_pred = moc.predict(X_train_tfidf)
micro_f1_train = f1_score(y_train, y_train_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance before optimizing - train data): {micro_f1_train:.4f}")

# Predict using the validation data
y_val_pred = moc.predict(X_val_tfidf)
micro_f1_val = f1_score(y_val, y_val_pred, average='micro')
print(f"Micro-Averaged F1-Score (Overall Performance before optimizing - validation data): {micro_f1_val:.4f}")

# Show overfitting
print(f"Overfitting before optimizing model: {micro_f1_train-micro_f1_val:.4f}")

# Predict with test data
y_test_pred = moc.predict(X_test_tfidf)
micro_f1_test = f1_score(y_test, y_test_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance before optimizing - test data): {micro_f1_test:.4f}")

param_grid = {
    # The 'estimator__C' targets the C parameter of the base LogisticRegression estimator
    'estimator__C': [8, 9, 10, 11, 12 ],
    # Optional: Test different regularization penalties (L1 for feature selection, L2 for general)
    'estimator__penalty': ['l1', 'l2'] 
}

grid_search = GridSearchCV(
    estimator=moc,
    param_grid=param_grid,
    scoring=micro_f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)

print("\nOptimizing logistic regression...")
grid_search.fit(X_train_tfidf, y_train)
print(f"\nBest Micro F1 Score: {grid_search.best_score_:.4f}")
print(f"Best Parameters found: {grid_search.best_params_}")

base_lr_final = LogisticRegression(
    C=grid_search.best_params_['estimator__C'],
    penalty=grid_search.best_params_['estimator__penalty'],
    solver='liblinear',        # Using liblinear/saga (adjust based on your best solver)
    random_state=42,
    max_iter=50000,
    class_weight='balanced'    # Recommended to keep this for imbalance compensation
)

final_model_logistic_regression = MultiOutputClassifier(base_lr_final, n_jobs=-1)
final_model_logistic_regression.fit(X_train_tfidf, y_train)

# Predict using the train data
y_train_pred = final_model_logistic_regression.predict(X_train_tfidf)
micro_f1_train = f1_score(y_train, y_train_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance after optimizing - train data): {micro_f1_train:.4f}")

# Predict using the validation data
y_val_pred = final_model_logistic_regression.predict(X_val_tfidf)
micro_f1_val = f1_score(y_val, y_val_pred, average='micro')
print(f"Micro-Averaged F1-Score (Overall Performance after optimizing - validation data): {micro_f1_val:.4f}")

# Show overfitting
print(f"Overfitting after optimizing model: {micro_f1_train-micro_f1_val:.4f}")

# Predict with test data
y_test_pred = final_model_logistic_regression.predict(X_test_tfidf)
micro_f1_test = f1_score(y_test, y_test_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance after optimizing - test data): {micro_f1_test:.4f}")

# Show final report
report = classification_report(
    y_val, 
    y_val_pred, 
    target_names=target_cols, 
    zero_division=0
)

print("Classification Report:")
print(report)


# 4. Train naive bayes
print("\n*** Train Naïve Bayes ***")
micro_f1_scorer = make_scorer(f1_score, average='micro', zero_division=0)

base_mnb = ComplementNB()
moc = MultiOutputClassifier(base_mnb, n_jobs=-1)

moc.fit(X_train_tfidf, y_train)

# Predict using the train data
y_train_pred = moc.predict(X_train_tfidf)
micro_f1_train = f1_score(y_train, y_train_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance before optimizing - train data): {micro_f1_train:.4f}")

# Predict using the validation data
y_val_pred = moc.predict(X_val_tfidf)
micro_f1_val = f1_score(y_val, y_val_pred, average='micro')
print(f"Micro-Averaged F1-Score (Overall Performance before optimizing - validation data): {micro_f1_val:.4f}")

# Show overfitting
print(f"Overfitting before optimizing model: {micro_f1_train-micro_f1_val:.4f}")

# Predict with test data
y_test_pred = moc.predict(X_test_tfidf)
micro_f1_test = f1_score(y_test, y_test_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance before optimizing - test data): {micro_f1_test:.4f}")

param_grid_mnb = {
    'estimator__alpha': [1.0, 0.5, 0.1, 0.01] 
}

grid_search = GridSearchCV(
    estimator=moc,
    param_grid=param_grid_mnb,
    scoring=micro_f1_scorer,
    cv=3,
    verbose=1,
    n_jobs=-1
)

print("Optimizing Naïve Bayes...")
grid_search.fit(X_train_tfidf, y_train) 
print(f"\nBest Micro F1 Score: {grid_search.best_score_:.4f}")
print(f"Best MNB Alpha: {grid_search.best_params_}")

base_mnb_final = ComplementNB(alpha=grid_search.best_params_['estimator__alpha'])
final_model_naive_bayes = MultiOutputClassifier(base_mnb_final, n_jobs=-1)
final_model_naive_bayes.fit(X_train_tfidf, y_train)

# Predict using the train data
y_train_pred = final_model_naive_bayes.predict(X_train_tfidf)
micro_f1_train = f1_score(y_train, y_train_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance after optimizing - train data): {micro_f1_train:.4f}")

# Predict using the validation data
y_val_pred = final_model_naive_bayes.predict(X_val_tfidf)
micro_f1_val = f1_score(y_val, y_val_pred, average='micro')
print(f"Micro-Averaged F1-Score (Overall Performance after optimizing - validation data): {micro_f1_val:.4f}")

# Show overfitting
print(f"Overfitting after optimizing model: {micro_f1_train-micro_f1_val:.4f}")

# Predict with test data
y_test_pred = final_model_naive_bayes.predict(X_test_tfidf)
micro_f1_test = f1_score(y_test, y_test_pred, average='micro')
print(f"\nMicro-Averaged F1-Score (Overall Performance after optimizing - test data): {micro_f1_test:.4f}")

report = classification_report(
    y_val, 
    y_val_pred, 
    target_names=target_cols, 
    zero_division=0
)

print("Classification Report:")
print(report)

# 6. Save models
joblib.dump(final_model_logistic_regression, f"{model_path}/{logistic_regression_pkl}")
joblib.dump(final_model_naive_bayes, f"{model_path}/{naive_bayes_pkl}")
joblib.dump(tfidf_vectorizer, f"{model_path}/{vectorizer_pkl}")

print("\nModels trained!")


[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Cleaning text...

*** Train base Logistic Regression ***

Micro-Averaged F1-Score (Overall Performance before optimizing - train data): 0.6837
Micro-Averaged F1-Score (Overall Performance before optimizing - validation data): 0.4989
Overfitting before optimizing model: 0.1848

Micro-Averaged F1-Score (Overall Performance before optimizing - test data): 0.5799

Optimizing logistic regression...
Fitting 3 folds for each of 10 candidates, totalling 30 fits

Best Micro F1 Score: 0.4570
Best Parameters found: {'estimator__C': 10, 'estimator__penalty': 'l1'}

Micro-Averaged F1-Score (Overall Performance after optimizing - train data): 0.7376
Micro-Averaged F1-Score (Overall Performance after optimizing - validation data): 0.4576
Overfitting after optimizing model: 0.2800

Micro-Averaged F1-Score (Overall Performance after optimizing - test data): 0.5236
Classification Report:
               precision    recall  f1-score   support

      IsToxic       0.59      0.57      0.58        72
    Is